In [11]:
from pathlib import Path
import time
import pandas as pd

RACINE = next(
    chemin
    for chemin in (Path.cwd(), *Path.cwd().parents)
    if (chemin / "pyproject.toml").is_file()
)

FICHIER_SOURCE = (
    RACINE / "data" / "raw" / "date=2026-11-02"
    / "lignes_commande.csv"
)

DOSSIER_SORTIE = (
    RACINE / "data" / "formats" / "date=2026-11-02"
)

DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)

print("Fichier source :", FICHIER_SOURCE)
print("Fichier existant :", FICHIER_SOURCE.exists())
print("Dossier de sortie :", DOSSIER_SORTIE)

Fichier source : c:\Users\hp\Desktop\senmarket\data\raw\date=2026-11-02\lignes_commande.csv
Fichier existant : True
Dossier de sortie : c:\Users\hp\Desktop\senmarket\data\formats\date=2026-11-02


In [12]:
df = pd.read_csv(FICHIER_SOURCE)

# Convertir la date pour préserver son type dans Parquet
if "date_commande" in df.columns:
    df["date_commande"] = pd.to_datetime(
        df["date_commande"],
        errors="raise"
    )

print("Dimensions :", df.shape)
display(df.head())
display(df.dtypes)

Dimensions : (200000, 6)


,ligne_commande_id,commande_id,produit_id,quantite,prix_unitaire_xof,montant_xof
0,LIG0000001,CMD0017828,PRD0000017,2,74109,148218
1,LIG0000002,CMD0029882,PRD0000003,2,131286,262572
2,LIG0000003,CMD0008286,PRD0000001,1,436863,436863
3,LIG0000004,CMD0071199,PRD0000016,2,309081,618162
4,LIG0000005,CMD0036770,PRD0000021,4,224839,899356


ligne_commande_id      str
commande_id            str
produit_id             str
quantite             int64
prix_unitaire_xof    int64
montant_xof          int64
dtype: object

In [13]:
# CSV
df.to_csv(
    DOSSIER_SORTIE / "lignes_commande.csv",
    index=False,
    encoding="utf-8"
)

# JSON classique : tableau d'objets
df.to_json(
    DOSSIER_SORTIE / "lignes_commande.json",
    orient="records",
    date_format="iso",
    force_ascii=False,
    indent=2
)

# JSON Lines : un objet JSON par ligne
df.to_json(
    DOSSIER_SORTIE / "lignes_commande.jsonl",
    orient="records",
    lines=True,
    date_format="iso",
    force_ascii=False
)

# Parquet
df.to_parquet(
    DOSSIER_SORTIE / "lignes_commande.parquet",
    index=False,
    engine="pyarrow"
)

print("Export des quatre formats terminé.")

Export des quatre formats terminé.


In [14]:
fichiers = {
    "CSV": DOSSIER_SORTIE / "lignes_commande.csv",
    "JSON": DOSSIER_SORTIE / "lignes_commande.json",
    "JSON Lines": DOSSIER_SORTIE / "lignes_commande.jsonl",
    "Parquet": DOSSIER_SORTIE / "lignes_commande.parquet"
}

lecteurs = {
    "CSV": lambda: pd.read_csv(fichiers["CSV"]),
    "JSON": lambda: pd.read_json(
        fichiers["JSON"],
        orient="records",
        dtype=False
    ),
    "JSON Lines": lambda: pd.read_json(
        fichiers["JSON Lines"],
        orient="records",
        lines=True,
        dtype=False
    ),
    "Parquet": lambda: pd.read_parquet(
        fichiers["Parquet"],
        engine="pyarrow"
    )
}

In [15]:
resultats = []

for format_nom, lecteur in lecteurs.items():

    temps = []

    for i in range(6):
        debut = time.perf_counter()

        donnees = lecteur()

        fin = time.perf_counter()
        temps.append(fin - debut)

    temps_mesures = temps[1:]

    temps_median = pd.Series(temps_mesures).median()

    taille_octets = fichiers[format_nom].stat().st_size
    taille_mo = taille_octets / (1024 ** 2)

    resultats.append({
        "Format": format_nom,
        "Temps médian (s)": round(temps_median, 4),
        "Taille (Mo)": round(taille_mo, 2),
        "Nombre de lignes": len(donnees)
    })

tableau_resultats = pd.DataFrame(resultats)

display(tableau_resultats)

,Format,Temps médian (s),Taille (Mo),Nombre de lignes
0,CSV,0.0269,9.53,200000
1,JSON,0.7528,35.28,200000
2,JSON Lines,0.9525,28.41,200000
3,Parquet,0.0294,4.32,200000


In [16]:
comparaison_types = {}

for format_nom, lecteur in lecteurs.items():

    donnees = lecteur()

    comparaison_types[format_nom] = donnees.dtypes.astype(str)

tableau_types = pd.DataFrame(comparaison_types)

display(tableau_types)

,CSV,JSON,JSON Lines,Parquet
ligne_commande_id,str,str,str,str
commande_id,str,str,str,str
produit_id,str,str,str,str
quantite,int64,int64,int64,int64
prix_unitaire_xof,int64,int64,int64,int64
montant_xof,int64,int64,int64,int64


In [17]:
chunks = []

for morceau in pd.read_csv(
    fichiers["CSV"],
    chunksize=10000
):
    chunks.append(morceau)

df_chunks = pd.concat(chunks, ignore_index=True)

df_complet = pd.read_csv(fichiers["CSV"])

pd.testing.assert_frame_equal(
    df_complet,
    df_chunks
)

print("Vérification réussie : lecture complète et lecture par morceaux identiques.")
print("Lecture complète :", len(df_complet), "lignes")
print("Lecture par morceaux :", len(df_chunks), "lignes")

Vérification réussie : lecture complète et lecture par morceaux identiques.
Lecture complète : 200000 lignes
Lecture par morceaux : 200000 lignes


In [18]:
for fichier in DOSSIER_SORTIE.iterdir():

    if fichier.is_file():
        print(
            f"{fichier.name} : "
            f"{fichier.stat().st_size / (1024 ** 2):.2f} Mo"
        )

lignes_commande.csv : 9.53 Mo
lignes_commande.json : 35.28 Mo
lignes_commande.jsonl : 28.41 Mo
lignes_commande.parquet : 4.32 Mo


In [19]:
fichier_commandes = (
    RACINE / "data" / "raw" / "date=2026-11-02"
    / "commandes.csv"
)

df_commandes = pd.read_csv(fichier_commandes)

print("Type initial de date_commande :")
print(df_commandes["date_commande"].dtype)

print("\nExemple de valeurs :")
display(df_commandes["date_commande"].head())

Type initial de date_commande :
str

Exemple de valeurs :


0    2026-05-22
1    2026-05-02
2    2026-03-08
3    2026-10-15
4    2026-04-01
Name: date_commande, dtype: str

In [2]:
import pandas as pd
from pathlib import Path

DATE = "2026-11-02"

RACINE = next(
    chemin
    for chemin in (Path.cwd(), *Path.cwd().parents)
    if (chemin / "pyproject.toml").is_file()
)
RAW = RACINE / "data" / "raw" / f"date={DATE}"

lignes = pd.read_csv(RAW / "lignes_commande.csv")
clients = pd.read_csv(RAW / "clients.csv")
commandes = pd.read_csv(RAW / "commandes.csv")

print("Lignes de commande :", len(lignes))
print("Clients :", len(clients))
print("Commandes :", len(commandes))

display(lignes.head())

Lignes de commande : 200000
Clients : 66666
Commandes : 80000


,ligne_commande_id,commande_id,produit_id,quantite,prix_unitaire_xof,montant_xof
0,LIG0000001,CMD0017828,PRD0000017,2,74109,148218
1,LIG0000002,CMD0029882,PRD0000003,2,131286,262572
2,LIG0000003,CMD0008286,PRD0000001,1,436863,436863
3,LIG0000004,CMD0071199,PRD0000016,2,309081,618162
4,LIG0000005,CMD0036770,PRD0000021,4,224839,899356


In [3]:
q1 = lignes["prix_unitaire_xof"].quantile(0.25)
q3 = lignes["prix_unitaire_xof"].quantile(0.75)

iqr = q3 - q1

borne_inferieure = q1 - 1.5 * iqr
borne_superieure = q3 + 1.5 * iqr

outliers = lignes[
    (lignes["prix_unitaire_xof"] < borne_inferieure)
    | (lignes["prix_unitaire_xof"] > borne_superieure)
]

print("Q1 :", q1)
print("Q3 :", q3)
print("IQR :", iqr)
print("Borne supérieure :", borne_superieure)
print("Nombre d'outliers détectés :", len(outliers))

display(outliers.head())

Q1 : 135947.75
Q3 : 453798.5
IQR : 317850.75
Borne supérieure : 930574.625
Nombre d'outliers détectés : 0


,ligne_commande_id,commande_id,produit_id,quantite,prix_unitaire_xof,montant_xof


In [4]:
valeurs_injectees = lignes[
    lignes["prix_unitaire_xof"] == 99_999_999
]

print("Valeurs injectées par le générateur :", len(valeurs_injectees))

Valeurs injectées par le générateur : 0


## Justification des décisions de nettoyage

### 1. Client sans ville

Une ville manquante ne signifie pas que le client est invalide.
Supprimer ce client ferait perdre des informations utiles comme
son historique de commandes.

La valeur est donc remplacée par « Inconnue ». Cette valeur permet
de conserver le client tout en distinguant une information absente
d'une ville réellement renseignée.

### 2. Montant de commande manquant

Le montant d'une ligne peut être recalculé à partir de la quantité
et du prix unitaire :

montant_xof = quantite × prix_unitaire_xof

Lorsque ces deux informations sont disponibles et valides,
le recalcul permet de conserver une ligne exploitable.

### 3. Prix aberrants

La méthode IQR permet de détecter les valeurs éloignées de la
distribution habituelle. Les prix supérieurs à 1 000 000 XOF
sont rejetés conformément à la règle métier définie pour SenMarket.

In [6]:
DOSSIER_REJETS = RACINE / "data" / "rejected" / f"date={DATE}"

bilan = pd.read_csv(DOSSIER_REJETS / "bilan_nettoyage.csv")

motifs = pd.read_csv(DOSSIER_REJETS / "motifs_rejet.csv")

display(bilan)
display(motifs)

,table,brut,acceptes,rejetes
0,clients,66666,66666,0
1,vendeurs,20,20,0
2,produits,40,40,0
3,commandes,80000,80000,0
4,lignes_commande,200000,200000,0
5,paiements,78400,78400,0
6,avis,4834,4834,0
7,navigation,400000,399984,16


,table,motif_rejet,nombre_lignes
0,navigation,date_future:date_evenement,16
